# Lab 17-10 · Building and Breaking OFDM

Six parts, each built on `ofdm.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
import ofdm as of
rng = np.random.default_rng(1710)

## Part 1 · Orthogonality

*Predict:* the normalized correlation of subcarriers one and two spacings apart over a window of 0.9 T.

In [2]:
t = np.linspace(0, 1, 100_001)
for frac in (1.0, 0.9):
    w = t <= frac
    for dk in (1, 2):
        c = abs(np.trapezoid(np.exp(2j * np.pi * dk * t[w]), t[w])) / frac
        print(f"window {frac:.1f} T, subcarriers {dk} apart: |correlation| = {c:.4f}")

window 1.0 T, subcarriers 1 apart: |correlation| = 0.0000
window 1.0 T, subcarriers 2 apart: |correlation| = 0.0000
window 0.9 T, subcarriers 1 apart: |correlation| = 0.1093
window 0.9 T, subcarriers 2 apart: |correlation| = 0.1039


## Part 2 · The cyclic prefix

*Predict:* which prefixes (0, 3, 5, 8 samples) give error-free 16-QAM through a 6-tap channel at 30 dB.

In [3]:
plt.figure()
h6 = np.array([0.7, 0.5, 0.3, 0.3, 0.2, 0.15]); h6 = h6 / np.linalg.norm(h6)
for cp in (0, 3, 5, 8):
    ser = of.simulate_link(64, cp, h6, 16, 30.0, 300, rng)
    print(f"prefix {cp} samples: symbol-error rate {ser:.2e}")
plt.plot(np.abs(of.channel_response(h6, 64)), "o-"); plt.xlabel("subcarrier"); plt.ylabel("|H_k|"); plt.title("the 6-tap channel per subcarrier"); plt.show()

prefix 0 samples: symbol-error rate 5.32e-02
prefix 3 samples: symbol-error rate 3.65e-04
prefix 5 samples: symbol-error rate 0.00e+00
prefix 8 samples: symbol-error rate 0.00e+00


## Part 3 · Per-subcarrier quality

*Predict:* each subcarrier's QPSK symbol-error rate from $|H_k|^2 \cdot$ SNR on a channel with a deep notch at 20 dB.

In [4]:
plt.figure()
h = np.array([1, 0.95 * np.exp(1j * 0.3)])
n, nsym, snr = 64, 2000, 20.0
pts = of.qam(4)
X = pts[rng.integers(0, 4, (nsym, n))]
Y = of.ofdm_demod(of.apply_channel(of.ofdm_mod(X, 4), h, snr, rng), n, 4, nsym)
H = of.channel_response(h, n)
zf = of.nearest(Y / H, pts)
mm = Y * np.conj(H) / (abs(H) ** 2 + 10 ** (-snr / 10))
mm_unbiased = of.nearest(mm / (abs(H) ** 2 / (abs(H) ** 2 + 10 ** (-snr / 10))), pts)
ser_meas = np.mean(zf != X, axis=0)
g = abs(H) ** 2 * 10 ** (snr / 10)
ser_pred = 1 - (1 - norm.sf(np.sqrt(g))) ** 2
plt.semilogy(np.maximum(ser_meas, 1e-5), "o", label="measured"); plt.semilogy(ser_pred, label="predicted from |H_k|^2 SNR")
plt.xlabel("subcarrier"); plt.ylabel("QPSK SER"); plt.legend(); plt.show()
print("ZF and bias-corrected MMSE decisions identical:", bool(np.all(zf == mm_unbiased)))

ZF and bias-corrected MMSE decisions identical: True


## Part 4 · Frequency offset

*Predict:* the ICI for ε = 0.02, 0.05 and 0.1 on 256 subcarriers.

In [5]:
for eps in (0.02, 0.05, 0.1):
    meas = of.measure_ici(eps, 256, 300, rng)
    print(f"eps = {eps}: measured {10 * math.log10(meas):.2f} dB, exact {10 * math.log10(of.cfo_ici_power(eps, 256)):.2f} dB, (pi eps)^2/3 {10 * math.log10(of.ici_small_offset(eps)):.2f} dB")

eps = 0.02: measured -28.80 dB, exact -28.81 dB, (pi eps)^2/3 -28.81 dB
eps = 0.05: measured -20.87 dB, exact -20.86 dB, (pi eps)^2/3 -20.85 dB
eps = 0.1: measured -14.89 dB, exact -14.89 dB, (pi eps)^2/3 -14.83 dB


## Part 5 · PAPR

*Predict:* the PAPR at $10^{-3}$ for OFDM with 300 subcarriers, and how much DFT-spread saves.

In [6]:
plt.figure()
p_ofdm = of.ofdm_papr(300, 10_000, rng, nfft=1200)
p_dfts = of.dfts_papr(300, 10_000, rng)
print(f"OFDM: 99.9th percentile {np.quantile(p_ofdm, 0.999):.2f} dB (approximation {of.papr_at(1e-3, 300):.2f} dB)")
print(f"DFT-spread OFDM: 99.9th percentile {np.quantile(p_dfts, 0.999):.2f} dB")
g = np.linspace(4, 13, 100)
plt.semilogy(g, [np.mean(p_ofdm > x) for x in g], label="OFDM")
plt.semilogy(g, [np.mean(p_dfts > x) for x in g], label="DFT-spread")
plt.semilogy(g, of.papr_ccdf_theory(g, 300), ":", label="approximation")
plt.ylim(1e-4, 1); plt.xlabel("PAPR (dB)"); plt.ylabel("CCDF"); plt.legend(); plt.show()

OFDM: 99.9th percentile 11.40 dB (approximation 11.01 dB)
DFT-spread OFDM: 99.9th percentile 7.87 dB


## Part 6 · NR numerology

*Predict:* the slot length at each μ, and whether a subframe is exactly 1 ms.

In [7]:
for mu in range(5):
    tu, tcp = of.nr_symbol(mu)
    slot = of.nr_slot_duration(mu)
    print(f"mu = {mu}: spacing {15 * 2 ** mu:4d} kHz, T = {tu * 1e6:7.3f} us, CP = {tcp * 1e6:6.3f} us, slot = {slot * 1e6:7.2f} us, subframe = {slot * 2 ** mu * 1e3:.6f} ms")
print("guard band, 100 MHz at 30 kHz with 273 RB:", of.nr_guardband_khz(100, 273, 30), "kHz")

mu = 0: spacing   15 kHz, T =  66.667 us, CP =  4.688 us, slot = 1000.00 us, subframe = 1.000000 ms
mu = 1: spacing   30 kHz, T =  33.333 us, CP =  2.344 us, slot =  500.00 us, subframe = 1.000000 ms
mu = 2: spacing   60 kHz, T =  16.667 us, CP =  1.172 us, slot =  250.00 us, subframe = 1.000000 ms
mu = 3: spacing  120 kHz, T =   8.333 us, CP =  0.586 us, slot =  125.00 us, subframe = 1.000000 ms
mu = 4: spacing  240 kHz, T =   4.167 us, CP =  0.293 us, slot =   62.50 us, subframe = 1.000000 ms
guard band, 100 MHz at 30 kHz with 273 RB: 845.0 kHz
